In [2]:
import os
os.environ["CDSE_USERNAME"]='galen.richardson@uottawa.ca'
os.environ["CDSE_PASSWORD"]='Esarocks123!'

import os, time, random, datetime as dt, shutil, zipfile  # stdlib
from pathlib import Path  # paths
import requests  # http
import rasterio  # jp2
import geopandas as gpd  # vectors
import pandas as pd  # tables

### Getting MGRS List from Shapefile

In [3]:
CAT="https://catalogue.dataspace.copernicus.eu/odata/v1"  # CDSE OData catalogue
TOK_URL="https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token"  # token endpoint
# ------------------------- patience / waits -------------------------
def _sleep_try(i, base=4.0, cap=300.0): time.sleep(min(cap, base*(2**i))*(0.6+0.8*random.random()))  # backoff+jitter
def _sleep_ok(lo=0.3, hi=1.2): time.sleep(lo+(hi-lo)*random.random())  # gentle throttle
# ------------------------- helpers -------------------------
def _ptype(level):  # L1C/L2A -> productType
    lvl=level.upper().replace("-","")  # normalize
    if lvl=="L1C": return "S2MSI1C"  # L1C
    if lvl in ("L2A","L2"): return "S2MSI2A"  # L2A
    raise ValueError("level must be 'L1C' or 'L2A'")  # guard
def _isoz(t): return t.replace(tzinfo=dt.timezone.utc).isoformat().replace("+00:00","Z")  # dt->Zulu
def _day_bounds(day): d=dt.date.fromisoformat(day); t0=dt.datetime(d.year,d.month,d.day,tzinfo=dt.timezone.utc); return _isoz(t0), _isoz(t0+dt.timedelta(days=1))  # [start,end)
def _month_bounds(y,m): y=int(y); m=int(m); t0=dt.datetime(y,m,1,tzinfo=dt.timezone.utc); t1=(dt.datetime(y+1,1,1,tzinfo=dt.timezone.utc) if m==12 else dt.datetime(y,m+1,1,tzinfo=dt.timezone.utc)); return _isoz(t0), _isoz(t1)  # [start,end)
def _months_to_list(months):  # normalize months
    if months is None: return list(range(1,13))  # all
    ms=[int(m) for m in months if 1<=int(m)<=12]  # clean
    return (list(range(ms[0], ms[1]+1)) if len(ms)==2 and ms[0]<=ms[1] else sorted(set(ms)))  # [5,9]->range else list
def _years_to_list(years):  # normalize years
    if years is None: raise ValueError("years must be provided")  # guard
    return sorted(set(int(y) for y in years))  # unique sorted
def _cloud_from_attrs(attrs):  # parse cloudCover
    if not attrs: return float("inf")  # missing
    if isinstance(attrs, dict): attrs=(attrs.get("OData.CSC.DoubleAttribute") or attrs.get("DoubleAttribute") or [])  # dict form
    for a in attrs:  # iterate
        if a.get("Name")=="cloudCover":
            try: return float(a.get("Value"))  # value
            except: return float("inf")  # bad parse
    return float("inf")  # not found
def _rm(p):  # delete file/dir best-effort
    try:
        p=Path(p)  # cast
        shutil.rmtree(p, ignore_errors=True) if p.is_dir() else (p.unlink() if p.exists() else None)  # rm
    except: pass
def _search_day(tiles, day, max_cloud, level, top=200):  # search a single UTC day
    start, end = _day_bounds(day)  # [start,end)
    return _search_window(tiles, start, end, max_cloud, level, top)  # reuse window search
# ------------------------- token cache -------------------------
class _Tok:  # tiny token cache
    def __init__(self): self.t=None; self.exp=0  # token+expiry
    def get(self, refresh_s=120):  # refresh if expiring soon
        now=time.time()  # now
        if self.t and now < (self.exp-refresh_s): return self.t  # still valid
        u=os.environ["CDSE_USERNAME"]; p=os.environ["CDSE_PASSWORD"]  # creds
        r=requests.post(TOK_URL, data={"client_id":"cdse-public","grant_type":"password","username":u,"password":p}, timeout=60)  # auth
        r.raise_for_status(); j=r.json()  # parse
        self.t=j["access_token"]; self.exp=now+int(j.get("expires_in",600))  # cache
        return self.t  # token
    def bust(self): self.t=None; self.exp=0  # force refresh next time
# ------------------------- robust JSON GET -------------------------
def _get_json(url, params=None, timeout=60, tries=12):  # patient JSON GET
    last=None  # last error
    for i in range(int(tries)):  # retries
        try:
            r=requests.get(url, params=params, timeout=timeout)  # request
            if r.status_code in (429,500,502,503,504): last=RuntimeError(f"HTTP {r.status_code}: {r.text[:200]}"); _sleep_try(i); continue  # transient
            r.raise_for_status()  # hard errors
            try: return r.json()  # parse
            except: raise RuntimeError(f"Non-JSON response (HTTP {r.status_code}): {r.text[:200]}")  # html/empty
        except (requests.RequestException, RuntimeError) as e:
            last=e; _sleep_try(i); continue  # wait+retry
    raise last if last else RuntimeError("JSON GET failed")  # give up
# ------------------------- search -------------------------
def _search_window(tiles, start, end, max_cloud, level, top):  # search [start,end)
    tf=" or ".join([f"contains(Name,'_T{t}_')" for t in tiles])  # tile in name
    pf="(contains(Name,'S2A_') or contains(Name,'S2B_') or contains(Name,'S2C_'))"  # platform tags
    cloud="" if max_cloud is None else f" and Attributes/OData.CSC.DoubleAttribute/any(att:att/Name eq 'cloudCover' and att/OData.CSC.DoubleAttribute/Value le {float(max_cloud)})"  # optional cloud filter
    filt=("Collection/Name eq 'SENTINEL-2' and "  # base
          f"Attributes/OData.CSC.StringAttribute/any(att:att/Name eq 'productType' and att/OData.CSC.StringAttribute/Value eq '{_ptype(level)}')"
          f"{cloud} and ContentDate/Start ge {start} and ContentDate/Start lt {end} and {pf} and ({tf})")  # full filter
    j=_get_json(f"{CAT}/Products", params={"$filter":filt,"$top":int(top),"$expand":"Attributes"}, timeout=60, tries=10)  # robust query
    vals=j.get("value", [])  # hits
    out=[{"Id":v["Id"],"Name":v["Name"],"Cloud":_cloud_from_attrs(v.get("Attributes")),"Start":v["ContentDate"]["Start"]} for v in vals]  # flatten
    return sorted(out, key=lambda r:(r["Cloud"], r["Start"]))  # best first
# ------------------------- download -------------------------
def _download_one(session, tokmgr, pid, name, outdir, tries=14, max_redirects=12):  # robust download
    outdir=Path(outdir); outdir.mkdir(parents=True, exist_ok=True)  # ensure dir
    fp=outdir/f"{name}.zip"  # final path
    if fp.exists() and zipfile.is_zipfile(fp): return str(fp), True  # skip good zip
    if fp.exists(): _rm(fp)  # delete corrupt/non-zip
    base=f"{CAT}/Products({pid})/$value"  # base URL
    last=None  # last error
    for i in range(int(tries)):  # attempts
        url=base  # reset URL each attempt
        try:
            for _ in range(int(max_redirects)+1):  # redirect chain
                tok=tokmgr.get()  # token (refreshable)
                r=session.get(url, headers={"Authorization":f"Bearer {tok}"}, stream=True, allow_redirects=False, timeout=300)  # request
                if r.status_code==401: tokmgr.bust(); raise requests.HTTPError("401 Unauthorized", response=r)  # refresh token
                if r.status_code in (429,500,502,503,504): raise requests.HTTPError(f"HTTP {r.status_code}", response=r)  # transient
                if r.status_code in (301,302,303,307,308):  # redirect
                    nxt=r.headers.get("Location");  # next
                    if not nxt: raise requests.HTTPError("Redirect without Location", response=r)  # guard
                    url=nxt; continue  # follow
                r.raise_for_status()  # hard errors

                part=fp.with_suffix(".zip.part")  # temp file
                with open(part,"wb") as f:  # write stream
                    for ch in r.iter_content(1024*1024):  # 1MB chunks
                        if ch: f.write(ch)  # write
                part.replace(fp)  # atomic finalize
                if not zipfile.is_zipfile(fp): raise zipfile.BadZipFile("Downloaded file is not a zip")  # catch HTML/partial
                _sleep_ok(0.4, 2.0)  # throttle success
                return str(fp), False  # done
            raise RuntimeError("Too many redirects")  # guard
        except (requests.RequestException, RuntimeError, zipfile.BadZipFile) as e:
            last=e; _rm(fp.with_suffix(".zip.part")); _rm(fp); _sleep_try(i, base=6.0, cap=600.0)  # cleanup + more patience
    raise last if last else RuntimeError("Download failed")  # give up
# ------------------------- validate (B02 zeros) -------------------------
def _find_safe(root): safes=list(Path(root).rglob("*.SAFE")); return safes[0] if safes else None  # find SAFE
def _pick_b2_jp2(safe_dir, level):  # choose B02
    safe_dir=Path(safe_dir); lvl=level.upper().replace("-","")  # cast
    pats=(["*B02*10m*.jp2","*_B02_10m.jp2","*B02*.jp2"] if lvl in ("L2A","L2") else ["*B02*.jp2"])  # patterns
    for pat in pats:
        hits=sorted(safe_dir.rglob(pat));  # find
        if hits: return hits[0]  # first
    return None  # none
def _count_zeros_jp2(jp2_path):  # blockwise count
    with rasterio.open(jp2_path) as src:
        n0=0
        for _, w in src.block_windows(1):
            a=src.read(1, window=w, masked=False)
            n0 += int((a==0).sum())
    return n0
def _validate_zip_b2zeros(zip_fp, level, b2_0max, tmp_root, redownload_once=None):  # extract+count; retry badzip once
    zip_fp=Path(zip_fp); tmp_root=Path(tmp_root)  # cast
    work=tmp_root/f"_tmp_safe_check_{zip_fp.stem}"; _rm(work); work.mkdir(parents=True, exist_ok=True)  # unique tmp
    def _extract():  # extract helper
        with zipfile.ZipFile(zip_fp, "r") as z: z.extractall(work)  # extract
    try:
        try: _extract()  # first try
        except zipfile.BadZipFile:
            if callable(redownload_once):  # allowed one redo
                _rm(zip_fp); redownload_once()  # delete + re-download
                _rm(work); work.mkdir(parents=True, exist_ok=True)  # reset
                try: _extract()  # second try
                except zipfile.BadZipFile: _rm(zip_fp); return False, None, "BadZipFile twice -> skipped (deleted zip)"  # give up
            else: _rm(zip_fp); return False, None, "BadZipFile -> deleted zip"  # fail
        safe=_find_safe(work)  # SAFE
        if safe is None: _rm(zip_fp); return False, None, "No .SAFE found (deleted zip)"  # fail
        jp2=_pick_b2_jp2(safe, level)  # B02
        if jp2 is None: _rm(zip_fp); return False, None, "No B02 .jp2 found (deleted zip)"  # fail
        try: n0=_count_zeros_jp2(jp2)  # count zeros
        except Exception as e: _rm(zip_fp); return False, None, f"JP2 read failed ({type(e).__name__}): {e} (deleted zip)"  # fail
        ok=(n0<=int(b2_0max))  # pass?
        if ok: _rm(safe)  # keep zip, remove SAFE
        else: _rm(zip_fp)  # delete bad zip
        return ok, int(n0), str(jp2)  # return
    finally:
        _rm(work)  # cleanup
# ------------------------- main -------------------------
def cdse_download_nearest_s2(tiles, date=None, max_cloud=None, months=None, years=None, level="L2A", n_images=None, download_dir="cdse_dl", b2_0max=1000, lookback_days=3650):  # months/years ignored
    tiles=[str(t).strip() for t in tiles if str(t).strip()]  # clean tiles
    if not tiles: raise ValueError("tiles list is empty")  # guard
    download_dir=Path(download_dir); download_dir.mkdir(parents=True, exist_ok=True)  # ensure dir
    print(f"CDSE | tiles={len(tiles)} | date={date} | level={level} | max_cloud={max_cloud} | n_images={n_images} | B2_zeros<={b2_0max} | lookback_days={lookback_days}")  # log

    attempts=[]; valid=[]  # records
    tokmgr=_Tok()  # token manager
    ses=requests.Session()  # session

    def _try_hit(tile, h, tag, seen):  # download+validate one hit
        if h["Id"] in seen: return False  # dup
        seen.add(h["Id"])  # mark
        cloud=("NA" if h["Cloud"]==float("inf") else f"{h['Cloud']:.1f}%")  # fmt
        print(f"  -> {tile} candidate cloud={cloud}  {tag}  {h['Name']}")  # log
        try: zip_fp, skipped=_download_one(ses, tokmgr, h["Id"], h["Name"], download_dir)  # download
        except Exception as e:
            print(f"     DL ERROR: {type(e).__name__}: {e}")  # log
            attempts.append({**h,"Tile":tile,"Tag":tag,"Path":None,"Skipped":False,"Valid":False,"B2_zeros":None,"Detail":f"download_error:{e}"})  # rec
            return False  # fail
        redo=lambda: _download_one(ses, tokmgr, h["Id"], h["Name"], download_dir)[0]  # redownload once if badzip
        ok, n0, detail=_validate_zip_b2zeros(zip_fp, level, b2_0max, download_dir, redownload_once=redo)  # validate (+ retry)
        attempts.append({**h,"Tile":tile,"Tag":tag,"Path":zip_fp,"Skipped":skipped,"Valid":bool(ok),"B2_zeros":n0,"Detail":detail})  # rec
        if ok: print(f"     VALID (B2_zeros={n0}) -> kept zip, removed SAFE"); valid.append(h["Name"]); return True  # keep
        z="NA" if n0 is None else str(n0)  # fmt
        print(f"     INVALID (B2_zeros={z}) -> deleted zip | {detail}")  # log
        return False  # fail

    for tile in tiles:  # tiles
        print(f"\n=== TILE {tile} | target={('ALL' if n_images is None else n_images)} ===")  # header
        seen=set(); v0=len(valid); target=None if n_images is None else int(n_images)  # state

        if date is None:  # broad scan: recent history only
            end=dt.datetime.now(dt.timezone.utc); start=end-dt.timedelta(days=int(lookback_days))  # window
            hits=_search_window([tile], _isoz(start), _isoz(end), max_cloud, level, top=1000)  # hits
            print(f"{tile} {start.date()}..{end.date()}: {len(hits)} hits")  # log
            for h in hits:  # candidates
                if target is not None and (len(valid)-v0)>=target: break  # done
                _try_hit(tile, h, f"{start.date()}..{end.date()}", seen)  # attempt
        else:  # nearest-to-date search (restrict months/years only if provided)
            d0=dt.date.fromisoformat(date)  # parse
            ys=(None if years  is None else set(int(y) for y in (years  if isinstance(years ,(list,tuple,set)) else [years ])))  # years set/None
            ms=(None if months is None else set(int(m) for m in (months if isinstance(months,(list,tuple,set)) else [months])))  # months set/None
            for k in range(int(lookback_days)+1):  # expand outward (bounded)
                for sgn in ([0] if k==0 else [1,-1]):  # 0,+,-
                    if target is not None and (len(valid)-v0)>=target: break  # done
                    d=d0+dt.timedelta(days=sgn*k)  # candidate date
                    if ys is not None and d.year  not in ys: continue  # restrict years if provided
                    if ms is not None and d.month not in ms: continue  # restrict months if provided
                    day=d.isoformat()  # day string
                    hits=_search_day([tile], day, max_cloud, level, top=200)  # hits
                    if hits:
                        b=hits[0]["Cloud"]; best=("NA" if b==float("inf") else f"{b:.1f}%")  # best cloud fmt
                        print(f"{tile} {day}: {len(hits)} hits (best cloud={best})")  # log
                    for h in hits:
                        if target is not None and (len(valid)-v0)>=target: break  # done
                        _try_hit(tile, h, day, seen)  # attempt
                if target is not None and (len(valid)-v0)>=target: break  # done
        print(f"=== DONE TILE {tile} | valid_downloaded={len(valid)-v0} ===")  # footer
    print(f"\nDone ALL TILES. Valid={len(valid)} / Attempted={len(attempts)} | dir={download_dir}")  # summary
    return attempts, valid  # return

In [ ]:
#Key West
#Need to REMOVE PRE-IRMA and those after 7/21/2021
Tiles=["17RMH"]
Date="2019-01-21"
Years=[2016,2017,2018,2019,2020,2021]
Maxcloud=10
Level="L2A"
N_images=100
Download_dir=r"G:\SDB_Compare_processing\Key West"
B2_0max=30000000
attempts, valid = cdse_download_nearest_s2(tiles=Tiles, date=Date, max_cloud=Maxcloud, level=Level,
                                           n_images=N_images, years=Years, download_dir=Download_dir,b2_0max=B2_0max)
valid

In [5]:
#Corsica
# Since there were 3 duplicates, we added the next three closest in date images
Tiles=["32TNL"]
Date="2018-04-04"
Maxcloud=10
Level="L2A"
Years=[2015,2016,2017,2018,2019,2020]
N_images=100
Download_dir=r"G:\SDB_Compare_processing\Corsica"
B2_0max=1000000000000000000
attempts, valid = cdse_download_nearest_s2(tiles=Tiles, date=Date, max_cloud=Maxcloud, level=Level,
                                           n_images=N_images, download_dir=Download_dir,b2_0max=B2_0max)
valid

CDSE | tiles=1 | date=2018-04-04 | level=L2A | max_cloud=10 | n_images=100 | B2_zeros<=1000000000000000000 | lookback_days=3650

=== TILE 32TNL | target=100 ===
32TNL 2018-04-02: 1 hits (best cloud=9.9%)
  -> 32TNL candidate cloud=9.9%  2018-04-02  S2A_MSIL2A_20180402T102021_N0500_R065_T32TNL_20230908T074848.SAFE
     VALID (B2_zeros=79501090) -> kept zip, removed SAFE
32TNL 2018-04-17: 1 hits (best cloud=7.6%)
  -> 32TNL candidate cloud=7.6%  2018-04-17  S2B_MSIL2A_20180417T102019_N0500_R065_T32TNL_20230910T103342.SAFE
     VALID (B2_zeros=79463120) -> kept zip, removed SAFE
32TNL 2018-04-19: 1 hits (best cloud=4.3%)
  -> 32TNL candidate cloud=4.3%  2018-04-19  S2A_MSIL2A_20180419T101031_N0500_R022_T32TNL_20230725T033931.SAFE
     VALID (B2_zeros=1) -> kept zip, removed SAFE
32TNL 2018-04-22: 1 hits (best cloud=1.5%)
  -> 32TNL candidate cloud=1.5%  2018-04-22  S2A_MSIL2A_20180422T102031_N0500_R065_T32TNL_20230915T161912.SAFE
     VALID (B2_zeros=79814506) -> kept zip, removed SAFE
32

     VALID (B2_zeros=79663076) -> kept zip, removed SAFE
32TNL 2017-10-04: 2 hits (best cloud=8.7%)
  -> 32TNL candidate cloud=8.7%  2017-10-04  S2A_MSIL2A_20171004T102021_N0500_R065_T32TNL_20230911T090542.SAFE
     VALID (B2_zeros=79441642) -> kept zip, removed SAFE
  -> 32TNL candidate cloud=8.8%  2017-10-04  S2A_MSIL2A_20171004T102021_N0500_R065_T32TNL_20231013T171543.SAFE
     VALID (B2_zeros=116299263) -> kept zip, removed SAFE
32TNL 2017-09-21: 1 hits (best cloud=0.1%)
  -> 32TNL candidate cloud=0.1%  2017-09-21  S2A_MSIL2A_20170921T101021_N0500_R022_T32TNL_20231014T182119.SAFE
     VALID (B2_zeros=159) -> kept zip, removed SAFE
32TNL 2018-10-21: 1 hits (best cloud=1.3%)
  -> 32TNL candidate cloud=1.3%  2018-10-21  S2B_MSIL2A_20181021T101039_N0500_R022_T32TNL_20230801T024759.SAFE
     VALID (B2_zeros=1) -> kept zip, removed SAFE
32TNL 2017-09-14: 1 hits (best cloud=1.6%)
  -> 32TNL candidate cloud=1.6%  2017-09-14  S2A_MSIL2A_20170914T102021_N0500_R065_T32TNL_20231003T175745.SAFE

     VALID (B2_zeros=79078860) -> kept zip, removed SAFE
32TNL 2017-05-29: 1 hits (best cloud=0.7%)
  -> 32TNL candidate cloud=0.7%  2017-05-29  S2B_MSIL2A_20170529T101029_N0500_R022_T32TNL_20231115T003547.SAFE
     VALID (B2_zeros=96) -> kept zip, removed SAFE
32TNL 2017-05-27: 1 hits (best cloud=0.0%)
  -> 32TNL candidate cloud=0.0%  2017-05-27  S2A_MSIL2A_20170527T102031_N0500_R065_T32TNL_20231112T000424.SAFE
     VALID (B2_zeros=79765352) -> kept zip, removed SAFE
32TNL 2019-02-13: 1 hits (best cloud=0.1%)
  -> 32TNL candidate cloud=0.1%  2019-02-13  S2A_MSIL2A_20190213T101131_N0500_R022_T32TNL_20221129T021433.SAFE
     VALID (B2_zeros=270) -> kept zip, removed SAFE
32TNL 2019-02-16: 1 hits (best cloud=0.7%)
  -> 32TNL candidate cloud=0.7%  2019-02-16  S2A_MSIL2A_20190216T102111_N0500_R065_T32TNL_20221125T222236.SAFE
     VALID (B2_zeros=78964209) -> kept zip, removed SAFE
32TNL 2017-05-17: 1 hits (best cloud=0.0%)
  -> 32TNL candidate cloud=0.0%  2017-05-17  S2A_MSIL2A_20170517T10

['S2A_MSIL2A_20180402T102021_N0500_R065_T32TNL_20230908T074848.SAFE',
 'S2B_MSIL2A_20180417T102019_N0500_R065_T32TNL_20230910T103342.SAFE',
 'S2A_MSIL2A_20180419T101031_N0500_R022_T32TNL_20230725T033931.SAFE',
 'S2A_MSIL2A_20180422T102031_N0500_R065_T32TNL_20230915T161912.SAFE',
 'S2A_MSIL2A_20180429T101031_N0500_R022_T32TNL_20230911T073353.SAFE',
 'S2A_MSIL2A_20180601T102021_N0500_R065_T32TNL_20230902T045008.SAFE',
 'S2A_MSIL2A_20180129T101251_N0500_R022_T32TNL_20230904T215802.SAFE',
 'S2B_MSIL2A_20180117T102339_N0500_R065_T32TNL_20230803T100345.SAFE',
 'S2B_MSIL2A_20180626T102019_N0500_R065_T32TNL_20230823T030042.SAFE',
 'S2A_MSIL2A_20180628T101031_N0500_R022_T32TNL_20230724T122838.SAFE',
 'S2A_MSIL2A_20180701T102021_N0500_R065_T32TNL_20230811T042458.SAFE',
 'S2B_MSIL2A_20180703T101029_N0500_R022_T32TNL_20230804T210048.SAFE',
 'S2A_MSIL2A_20180708T101031_N0500_R022_T32TNL_20230813T195422.SAFE',
 'S2A_MSIL2A_20180711T102021_N0500_R065_T32TNL_20230801T122120.SAFE',
 'S2B_MSIL2A_2018071

In [10]:
#Cape Cod
Tiles=["19TCG"]
Date="2018-07-03"
Maxcloud=10
Months=[3,4,5,6,7,8,9,10,11]
Years=[2016,2017,2018,2019,2020]
Level="L2A"
N_images=100
Download_dir=r"G:\SDB_Compare_processing\Cape Cod"
B2_0max=1000
attempts, valid = cdse_download_nearest_s2(tiles=Tiles, date=Date, max_cloud=Maxcloud, level=Level,
                                           n_images=N_images,months=Months, years=Years, download_dir=Download_dir,b2_0max=B2_0max)
valid

CDSE | tiles=1 | date=2018-07-03 | level=L2A | max_cloud=10 | n_images=100 | B2_zeros<=1000 | lookback_days=3650

=== TILE 19TCG | target=100 ===
19TCG 2018-07-02: 1 hits (best cloud=1.0%)
  -> 19TCG candidate cloud=1.0%  2018-07-02  S2B_MSIL2A_20180702T153809_N0500_R011_T19TCG_20230713T130236.SAFE
     INVALID (B2_zeros=63649392) -> deleted zip | G:\SDB_Compare_processing\Cape Cod\_tmp_safe_check_S2B_MSIL2A_20180702T153809_N0500_R011_T19TCG_20230713T130236.SAFE\S2B_MSIL2A_20180702T153809_N0500_R011_T19TCG_20230713T130236.SAFE\GRANULE\L2A_T19TCG_A006901_20180702T154544\IMG_DATA\R10m\T19TCG_20180702T153809_B02_10m.jp2
19TCG 2018-07-07: 1 hits (best cloud=0.4%)
  -> 19TCG candidate cloud=0.4%  2018-07-07  S2A_MSIL2A_20180707T153911_N0500_R011_T19TCG_20230809T073112.SAFE
     INVALID (B2_zeros=63460874) -> deleted zip | G:\SDB_Compare_processing\Cape Cod\_tmp_safe_check_S2A_MSIL2A_20180707T153911_N0500_R011_T19TCG_20230809T073112.SAFE\S2A_MSIL2A_20180707T153911_N0500_R011_T19TCG_20230809T

     INVALID (B2_zeros=63329522) -> deleted zip | G:\SDB_Compare_processing\Cape Cod\_tmp_safe_check_S2B_MSIL2A_20190309T154049_N0500_R011_T19TCG_20221119T095834.SAFE\S2B_MSIL2A_20190309T154049_N0500_R011_T19TCG_20221119T095834.SAFE\GRANULE\L2A_T19TCG_A010476_20190309T154955\IMG_DATA\R10m\T19TCG_20190309T154049_B02_10m.jp2
19TCG 2017-10-17: 1 hits (best cloud=3.3%)
  -> 19TCG candidate cloud=3.3%  2017-10-17  S2A_MSIL2A_20171017T153601_N0500_R111_T19TCG_20230907T094624.SAFE
     VALID (B2_zeros=38) -> kept zip, removed SAFE
19TCG 2019-04-25: 1 hits (best cloud=0.9%)
  -> 19TCG candidate cloud=0.9%  2019-04-25  S2B_MSIL2A_20190425T153609_N0500_R111_T19TCG_20221023T115432.SAFE
     VALID (B2_zeros=0) -> kept zip, removed SAFE
19TCG 2017-08-31: 1 hits (best cloud=4.3%)
  -> 19TCG candidate cloud=4.3%  2017-08-31  S2A_MSIL2A_20170831T153911_N0500_R011_T19TCG_20230830T194818.SAFE
     INVALID (B2_zeros=64548153) -> deleted zip | G:\SDB_Compare_processing\Cape Cod\_tmp_safe_check_S2A_MSIL2A_

     VALID (B2_zeros=115) -> kept zip, removed SAFE
19TCG 2019-11-16: 1 hits (best cloud=4.8%)
  -> 19TCG candidate cloud=4.8%  2019-11-16  S2A_MSIL2A_20191116T153601_N0500_R111_T19TCG_20230621T220744.SAFE
     VALID (B2_zeros=97) -> kept zip, removed SAFE
19TCG 2019-11-21: 1 hits (best cloud=2.9%)
  -> 19TCG candidate cloud=2.9%  2019-11-21  S2B_MSIL2A_20191121T153559_N0500_R111_T19TCG_20230624T203826.SAFE
     VALID (B2_zeros=9) -> kept zip, removed SAFE
19TCG 2019-11-26: 1 hits (best cloud=5.5%)
  -> 19TCG candidate cloud=5.5%  2019-11-26  S2A_MSIL2A_20191126T153601_N0500_R111_T19TCG_20230622T075037.SAFE
     VALID (B2_zeros=81) -> kept zip, removed SAFE
19TCG 2016-11-14: 2 hits (best cloud=0.7%)
  -> 19TCG candidate cloud=0.7%  2016-11-14  S2A_MSIL2A_20161114T154522_N0500_R011_T19TCG_20230925T184337.SAFE
     INVALID (B2_zeros=104572504) -> deleted zip | G:\SDB_Compare_processing\Cape Cod\_tmp_safe_check_S2A_MSIL2A_20161114T154522_N0500_R011_T19TCG_20230925T184337.SAFE\S2A_MSIL2A_2

     INVALID (B2_zeros=63392385) -> deleted zip | G:\SDB_Compare_processing\Cape Cod\_tmp_safe_check_S2B_MSIL2A_20200820T153819_N0500_R011_T19TCG_20230514T094031.SAFE\S2B_MSIL2A_20200820T153819_N0500_R011_T19TCG_20230514T094031.SAFE\GRANULE\L2A_T19TCG_A018055_20200820T154426\IMG_DATA\R10m\T19TCG_20200820T153819_B02_10m.jp2
19TCG 2020-09-04: 1 hits (best cloud=0.1%)
  -> 19TCG candidate cloud=0.1%  2020-09-04  S2A_MSIL2A_20200904T153911_N0500_R011_T19TCG_20230607T122343.SAFE
     INVALID (B2_zeros=64152808) -> deleted zip | G:\SDB_Compare_processing\Cape Cod\_tmp_safe_check_S2A_MSIL2A_20200904T153911_N0500_R011_T19TCG_20230607T122343.SAFE\S2A_MSIL2A_20200904T153911_N0500_R011_T19TCG_20230607T122343.SAFE\GRANULE\L2A_T19TCG_A027178_20200904T155003\IMG_DATA\R10m\T19TCG_20200904T153911_B02_10m.jp2
19TCG 2020-09-19: 1 hits (best cloud=7.7%)
  -> 19TCG candidate cloud=7.7%  2020-09-19  S2B_MSIL2A_20200919T153819_N0500_R011_T19TCG_20230603T082035.SAFE
     INVALID (B2_zeros=63125356) -> delete

['S2B_MSIL2A_20180719T153559_N0500_R111_T19TCG_20230802T031408.SAFE',
 'S2A_MSIL2A_20180405T153601_N0500_R111_T19TCG_20230818T084845.SAFE',
 'S2B_MSIL2A_20180331T153549_N0500_R111_T19TCG_20231102T213825.SAFE',
 'S2B_MSIL2A_20181017T153559_N0500_R111_T19TCG_20230814T024008.SAFE',
 'S2A_MSIL2A_20181022T153551_N0500_R111_T19TCG_20230802T173438.SAFE',
 'S2A_MSIL2A_20181111T153551_N0500_R111_T19TCG_20230815T040801.SAFE',
 'S2A_MSIL2A_20171017T153601_N0500_R111_T19TCG_20230907T094624.SAFE',
 'S2B_MSIL2A_20190425T153609_N0500_R111_T19TCG_20221023T115432.SAFE',
 'S2A_MSIL2A_20170709T153601_N0500_R111_T19TCG_20230921T013754.SAFE',
 'S2B_MSIL2A_20170704T153559_N0500_R111_T19TCG_20231011T192736.SAFE',
 'S2A_MSIL2A_20190719T153601_N0500_R111_T19TCG_20230805T235144.SAFE',
 'S2B_MSIL2A_20190922T153559_N0500_R111_T19TCG_20230703T235159.SAFE',
 'S2A_MSIL2A_20190927T153601_N0500_R111_T19TCG_20230619T181023.SAFE',
 'S2B_MSIL2A_20191101T153559_N0500_R111_T19TCG_20230801T171341.SAFE',
 'S2A_MSIL2A_2019110

In [4]:
#Ungava Bay
Tiles=["19VDG"]
Date="2018-10-15"
Maxcloud=20
Level="L2A"
N_images=100
Months=[7,8,9]
Years=[2016,2017,2018,2019,2020]
Download_dir=r"G:\SDB_Compare_processing\Ungava Bay"
B2_0max=70000000
attempts, valid = cdse_download_nearest_s2(tiles=Tiles, date=Date, max_cloud=Maxcloud, level=Level,
                                           n_images=N_images,months=Months, years=Years, download_dir=Download_dir,b2_0max=B2_0max)
valid

CDSE | tiles=1 | date=2018-10-15 | level=L2A | max_cloud=20 | n_images=100 | B2_zeros<=70000000 | lookback_days=3650

=== TILE 19VDG | target=100 ===
19VDG 2018-09-28: 1 hits (best cloud=3.1%)
  -> 19VDG candidate cloud=3.1%  2018-09-28  S2A_MSIL2A_20180928T155021_N0500_R054_T19VDG_20230730T060716.SAFE
     INVALID (B2_zeros=87407210) -> deleted zip | G:\SDB_Compare_processing\Ungava Bay\_tmp_safe_check_S2A_MSIL2A_20180928T155021_N0500_R054_T19VDG_20230730T060716.SAFE\S2A_MSIL2A_20180928T155021_N0500_R054_T19VDG_20230730T060716.SAFE\GRANULE\L2A_T19VDG_A017068_20180928T155454\IMG_DATA\R10m\T19VDG_20180928T155021_B02_10m.jp2
19VDG 2018-09-13: 1 hits (best cloud=6.9%)
  -> 19VDG candidate cloud=6.9%  2018-09-13  S2B_MSIL2A_20180913T154859_N0500_R054_T19VDG_20230803T073645.SAFE
     INVALID (B2_zeros=87305380) -> deleted zip | G:\SDB_Compare_processing\Ungava Bay\_tmp_safe_check_S2B_MSIL2A_20180913T154859_N0500_R054_T19VDG_20230803T073645.SAFE\S2B_MSIL2A_20180913T154859_N0500_R054_T19VDG_2

     INVALID (B2_zeros=88229560) -> deleted zip | G:\SDB_Compare_processing\Ungava Bay\_tmp_safe_check_S2B_MSIL2A_20200803T154819_N0500_R054_T19VDG_20230418T212442.SAFE\S2B_MSIL2A_20200803T154819_N0500_R054_T19VDG_20230418T212442.SAFE\GRANULE\L2A_T19VDG_A017812_20200803T155035\IMG_DATA\R10m\T19VDG_20200803T154819_B02_10m.jp2
19VDG 2020-08-06: 1 hits (best cloud=0.0%)
  -> 19VDG candidate cloud=0.0%  2020-08-06  S2B_MSIL2A_20200806T155829_N0500_R097_T19VDG_20230316T152250.SAFE
     VALID (B2_zeros=0) -> kept zip, removed SAFE
19VDG 2020-08-13: 2 hits (best cloud=0.0%)
  -> 19VDG candidate cloud=0.0%  2020-08-13  S2B_MSIL2A_20200813T154819_N0500_R054_T19VDG_20230317T114804.SAFE
     INVALID (B2_zeros=96777628) -> deleted zip | G:\SDB_Compare_processing\Ungava Bay\_tmp_safe_check_S2B_MSIL2A_20200813T154819_N0500_R054_T19VDG_20230317T114804.SAFE\S2B_MSIL2A_20200813T154819_N0500_R054_T19VDG_20230317T114804.SAFE\GRANULE\L2A_T19VDG_A017955_20200813T155628\IMG_DATA\R10m\T19VDG_20200813T154819_

['S2A_MSIL2A_20180822T155901_N0500_R097_T19VDG_20230822T181638.SAFE',
 'S2A_MSIL2A_20180815T161001_N0500_R140_T19VDG_20230811T010703.SAFE',
 'S2B_MSIL2A_20180807T155859_N0500_R097_T19VDG_20230803T052107.SAFE',
 'S2B_MSIL2A_20180728T155819_N0500_R097_T19VDG_20230805T013825.SAFE',
 'S2A_MSIL2A_20180726T160901_N0500_R140_T19VDG_20230722T070035.SAFE',
 'S2A_MSIL2A_20190817T155911_N0500_R097_T19VDG_20230712T204405.SAFE',
 'S2B_MSIL2A_20170723T155909_N0500_R097_T19VDG_20231005T193939.SAFE',
 'S2B_MSIL2A_20170703T155909_N0500_R097_T19VDG_20231005T144317.SAFE',
 'S2A_MSIL2A_20170701T161341_N0500_R140_T19VDG_20231007T140053.SAFE',
 'S2B_MSIL2A_20200707T155819_N0500_R097_T19VDG_20230614T051749.SAFE',
 'S2B_MSIL2A_20200710T160829_N0500_R140_T19VDG_20230603T070409.SAFE',
 'S2A_MSIL2A_20200715T160911_N0500_R140_T19VDG_20230424T154734.SAFE',
 'S2B_MSIL2A_20200717T155819_N0500_R097_T19VDG_20230425T192628.SAFE',
 'S2B_MSIL2A_20200806T155829_N0500_R097_T19VDG_20230316T152250.SAFE',
 'S2B_MSIL2A_2020081